# Brain Tumor Multi-task — Colab runner
Runtime → Change runtime type → **GPU**.

Google Drive layout (shared folder, path set in `DRIVE` below):
```
brain_tumor/
  raw/            4 zip files from Figshare (only needed once)
  processed.zip   created by this notebook the first time
  runs/           training outputs (checkpoints, metrics, figures)
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/brain_tumor'   # <- change to your shared folder
REPO = 'https://github.com/G9YzNDN/DLMED.git'

In [ ]:
!git clone -q $REPO /content/DLMED || (cd /content/DLMED && git pull -q)
%cd /content/DLMED
!pip install -q -r requirements.txt

## Data
Images are copied to the Colab disk (reading thousands of small files from Drive is slow).
The first run converts the `.mat` files and saves `processed.zip` to Drive; later runs just unzip it.
The split always comes from the committed `splits/splits.csv`.

In [ ]:
import os
if os.path.exists(f'{DRIVE}/processed.zip'):
    !mkdir -p data && unzip -q -o "$DRIVE/processed.zip" -d data
else:
    !mkdir -p data/raw && for z in "$DRIVE"/raw/*.zip; do unzip -q -o "$$z" -d data/raw; done
    !python src/prepare_data.py --raw_dir data/raw --out_dir data/processed
    !cd data && zip -q -r "$DRIVE/processed.zip" processed
!ls data/processed/images | wc -l
!cat splits/data_summary.csv

## Short trial (2 epochs, validation only)
An existing run folder is protected: use a new name for each trial. Published original results predate the augmentation-seed fix; new runs are separate experiments.
A: `--tasks cls` · B: `--tasks seg` · C: `--tasks both` · D: `--tasks both --weighting uncertainty`

In [ ]:
NAME = 'C_demo_seed0'  # choose a new name if this folder already exists
!python src/train.py --tasks both --epochs 2 --seed 0 --name $NAME --out_dir "$DRIVE/runs" 

In [ ]:
!python src/evaluate.py --run "$DRIVE/runs/$NAME" --data_dir data/processed --split val

In [ ]:
from IPython.display import Image, display
for f in ['confusion_matrix.png', 'seg_best_worst.png', 'misclassified.png']:
    p = f'{DRIVE}/runs/{NAME}/{f}'
    if os.path.exists(p):
        display(Image(p))

## All experiments (A–D × 3 seeds) and the comparison table
Takes several hours; can be split across team members by running single experiments above.

In [ ]:
# !bash scripts/run_experiments.sh --out_dir "$DRIVE/runs"
!python src/summarize.py --runs_dir "$DRIVE/runs" --split test